
# 練習問題: ガチャ (クーポンコレクター問題)

## 目標

「N 種類の景品が等確率で出るガチャを, **全種類そろうまで**引くと, 平均何回引くことになるか?」をモンテカルロ法で推定する。多数の独立な試行を `parallel for` で分担し, 結果を `reduction` で集計する, という並列計算の典型パターンを身につける。

## 課題

`gacha.cpp` (または `gacha.f90`) は, 1回の試行 (全種そろうまで引く) を行う関数 `one_trial` を持ち, それを `T` 回繰り返して引き回数の平均と標準偏差を求める。
各試行は**互いに独立** (それぞれ別の乱数列を使う) なので, 並列化できる。

`TODO` の箇所に **OpenMP の指示を追加** し, 試行ループを並列化して合計 (`total`) と二乗和 (`totalsq`) を集計せよ。

- C++: ループの直前に `#pragma omp parallel for reduction(+:total,totalsq)` を1行加える。
- Fortran: `do` ループを `!$omp parallel do reduction(+:total,totalsq)` と `!$omp end parallel do` で囲む。

`reduction(+:total,totalsq)` のように, 複数の変数を同時に縮約できる。

### 乱数と並列化の関係 (この問題の肝)

並列モンテカルロで厄介なのが乱数である。

- 1つの乱数生成器を全スレッドで共有すると, 内部状態の奪い合い (競合状態) になり壊れる。
- かといってスレッドごとに別の生成器を持たせると, スレッド数を変えるたびに「引かれる乱数の集合」が変わり, **答えがスレッド数によって変わって**しまう。

そこでこのコードは, **状態を持たない (カウンタベースの) 乱数**を使う。`draw(t, k, N)` は「t 回目の試行の k 回目の引き」という**番号だけ**から乱数を決める純粋関数で,

- どのスレッドが計算しても, (t, k) が同じなら必ず同じ値 → **引かれる乱数列はスレッド数によらず同一**。
- さらに引き回数は整数なので, `total` を整数で縮約すれば足す順番によらず **答えがビット単位で完全に一致**する。

実際, `OMP_NUM_THREADS` を 1, 2, 4, 8 と変えても出力が変わらないことを確認せよ。「並列化しても答えが1スレッドと完全に同じ」は, 並列プログラムが正しいことの強力な証拠になる。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore gacha.cpp -o gacha.exe

# Fortran
nvfortran -fast -mp=multicore gacha.f90 -o gacha.exe
```

第1引数で景品の種類数 `N`, 第2引数で試行回数 `T` を変えられる。

```
OMP_NUM_THREADS=4 ./gacha.exe 10 1000000
```

## 期待される結果

クーポンコレクター問題の理論では, 全種そろえるのに必要な平均回数は `N × H_N` (H_N = 1 + 1/2 + … + 1/N)。

- N=10 → 約 29.3 回
- N=50 → 約 224.9 回

```
N=10, trials=1000000: 平均 29.29 回 (理論 29.29), 標準偏差 11.2
```

- 試行回数 `T` を増やすほど, 推定した平均が理論値に近づく。
- **標準偏差が大きい** (上振れ・下振れが激しい) ことにも注目: 運が悪いと理論平均の倍以上引くこともある, というガチャの「沼」が数字で見える。
- `N` を変えて, 種類が増えると一気に大変になる (N に対しておよそ N·log N で増える) ことを確かめよ。

参考: スレッド数を増やすと (試行が独立なので) ほぼ比例して速くなる。台数効果も観察できる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ gacha.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* ── 状態を持たない (カウンタベースの) 乱数: たった1つの純粋関数 ──────────
   draw_rand(seed, k, N) は 0..N-1 の整数を返す。
   - seed は「どの乱数列 (ストリーム) を使うか」を選ぶ番号 (この問題では試行ごとに変える)。
   - k は「その列の何番目を取り出すか」。同じ seed でも k が違えば別の値。
   - 同じ (seed,k) なら必ず同じ値を返す純粋関数なので, どのスレッドが計算しても,
     引かれる乱数列はスレッド数によらず同一になる (共有状態が無いので競合もしない)。
   (教育用の簡単なハッシュ。M=2^31-1 未満で計算し, 途中の積も 64bit に収まる。)  */
static inline int draw_rand(long long seed, long long k, int N) {
  const long long M = 2147483647LL;   /* 2^31 - 1 */
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;   /* seed と k を1つにまとめる */
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (int)(x % N);
}

/* 1試行: N 種類が等確率で出るとき, 全種類そろうまでに引いた回数。
   そろった種類を 64bit のビットマスクで管理する (N <= 62 を想定)。
   seed に試行番号を渡し, k=0,1,2,... と引いていく。 */
long one_trial(int N, long long seed) {
  unsigned long long got = 0;
  unsigned long long full = (N == 64 ? ~0ULL : ((1ULL << N) - 1));
  long k = 0;
  while (got != full) {
    got |= (1ULL << draw_rand(seed, k, N));
    k++;
  }
  return k;   /* 引いた回数 */
}

int main(int argc, char ** argv) {
  int  N = (argc > 1 ? atoi(argv[1]) : 10);          /* 景品の種類数 */
  long T = (argc > 2 ? atol(argv[2]) : 1000000);     /* 試行回数 */
  /* 引き回数は整数なので整数で集計する → 足す順番によらず答えが完全に一致する */
  long long total = 0, totalsq = 0;

  /* T 回の試行は互いに独立。各試行の引き回数を集計する。 */
  double t0 = omp_get_wtime();
  // TODO: 各試行は独立なので #pragma omp parallel for reduction(+:total,totalsq) で並列化・集計せよ.
  for (long t = 0; t < T; t++) {
    long d = one_trial(N, t);
    total   += d;
    totalsq += (long long)d * d;
  }
  double elapsed = omp_get_wtime() - t0;

  double mean = (double)total / T;
  double var  = (double)totalsq / T - mean * mean;
  /* 理論期待値 = N * H_N (H_N は調和数) */
  double H = 0.0;
  for (int k = 1; k <= N; k++) H += 1.0 / k;
  printf("N=%d, trials=%ld: 平均 %.3f 回 (理論 %.3f), 標準偏差 %.3f\n",
         N, T, mean, N * H, sqrt(var));
  printf("elapsed = %.3f sec\n", elapsed);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore gacha.cpp -o gacha_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gacha_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ gacha.f90
module gacha_mod
contains
  ! 状態を持たない (カウンタベースの) 乱数: たった1つの純粋関数。
  ! draw_rand(seed, k, N) は 0..N-1 の整数を返す。
  ! - seed は「どの乱数列(ストリーム)を使うか」を選ぶ番号 (この問題では試行ごとに変える)。
  ! - k は「その列の何番目を取り出すか」。同じ seed でも k が違えば別の値。
  ! - 同じ (seed,k) なら必ず同じ値 → スレッドで分担しても乱数列はスレッド数によらず同一。
  ! (教育用の簡単なハッシュ。M=2^31-1 未満で計算し, 途中の積も 64bit に収まる。)
  function draw_rand(seed, k, N) result(idx)
    integer(8), intent(in) :: seed, k
    integer, intent(in) :: N
    integer :: idx
    integer(8), parameter :: M = 2147483647_8   ! 2^31 - 1
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)   ! seed と k を1つにまとめる
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    idx = int(modulo(x, int(N, 8)))
  end function draw_rand

  ! 1試行: 全種類そろうまでに引いた回数 (そろった種類を 64bit マスクで管理, N <= 62)
  ! seed に試行番号を渡し, k=0,1,2,... と引いていく。
  function one_trial(N, seed) result(draws)
    integer, intent(in) :: N
    integer(8), intent(in) :: seed
    integer(8) :: draws, got, full, k
    integer :: idx
    got = 0_8
    full = ishft(1_8, N) - 1_8
    k = 0_8
    do while (got /= full)
       idx = draw_rand(seed, k, N)
       got = ior(got, ishft(1_8, idx))
       k = k + 1_8
    end do
    draws = k
  end function one_trial
end module gacha_mod

program gacha
  use gacha_mod
  use omp_lib
  character(len=32) :: arg
  integer :: N, k
  integer(8) :: T, t_, total, totalsq, d
  real(8) :: mean, var, H, t0, elapsed
  N = 10
  T = 1000000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) T
  end if
  ! 引き回数は整数なので整数で集計する → 足す順番によらず答えが完全に一致する
  total = 0_8; totalsq = 0_8

  ! T 回の試行は互いに独立。各試行の引き回数を集計する。
  t0 = omp_get_wtime()
  ! TODO: 各試行は独立なので !$omp parallel do reduction(+:total,totalsq) で並列化・集計せよ.
  do t_ = 0, T - 1
     d = one_trial(N, t_)
     total = total + d
     totalsq = totalsq + d * d
  end do
  ! TODO: 上の parallel do を閉じる !$omp end parallel do を書け.
  elapsed = omp_get_wtime() - t0

  mean = real(total, 8) / T
  var  = real(totalsq, 8) / T - mean * mean
  ! 理論期待値 = N * H_N
  H = 0.0d0
  do k = 1, N
     H = H + 1.0d0 / k
  end do
  print "(a,i0,a,i0,a,f0.3,a,f0.3,a,f0.3)", &
       "N=", N, ", trials=", T, ": 平均 ", mean, " 回 (理論 ", N * H, "), 標準偏差 ", sqrt(var)
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program gacha

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore gacha.f90 -o gacha_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gacha_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:gacha.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:gacha.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:gacha.cpp}